# Space hotel ACOS application and aggregation

### Composizione del Dataset SPACE
Il file space_summ.json contiene per ogni riga un Hotel (con tutte le sue 100 recensioni nidificate dentro) <br>
Il file è strutturato come segue, per ogni hotel abbiamo:
* entity_id: identificativo univoco dell'hotel
* reviews: lista che contiene 100 oggetti 'review' (ogni review è una recensione dell'hotel)
* ogni review è composta da:
    * ranting: voto da 1 a 5 dato dall'utente all'hotel
    * review_id: identificativo univoco della recensione
    * sentences: lista di frasi che compongono la recensione


In [1]:
import torch
import json
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from acos_model import ModernBertACOS_Extractor, ModernBertACOSClassifier
import pickle
from acos_utils import predict_quadruples_space, get_spans
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from thefuzz import fuzz
from sentence_transformers import SentenceTransformer, util
import os
import ollama


# 1. SETUP DEVICE
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Device operativo: {device}")

Device operativo: mps


### Caricamento Dataset e Modelli

Questo codice esegue due operazioni cruciali:

**1. Caricamento del Dataset SPACE:**
- Legge il file JSON `space_summ.json` contenente 50 hotel con le loro 100 recensioni
- Itera su ogni hotel e estrae le 100 recensioni associate
- Per ogni recensione, unisce le frasi in un testo continuo
- Crea un DataFrame con tre colonne: `hotel_id`, `review_id`, `text`
- Stampa statistiche: numero di hotel unici e totale di recensioni

**2. Caricamento dei Modelli ACOS (2-Step Pipeline):**
- Carica il tokenizer `ModernBERT-base` per preprocessare il testo
- **Step 1 (Extractor)**: Modello che estrae i **token di aspetti e opinioni** usando BIO tagging (5 label: O, B-ASP, I-ASP, B-OPI, I-OPI)
- **Step 2 (Classifier)**: Modello che assegna a ogni aspetto una **categoria** (13 categorie per hotel) e il **sentiment** (positivo/negativo)
- Entrambi i modelli vengono messi in modalità `eval()` pronta per l'inferenza

Al termine, il pipeline è pronto per estrarre le quadruple ACOS dalle recensioni.

In [3]:
# 2. CARICAMENTO DEL DATASET SPACE (space_summ.json)
print("\nCaricamento dataset SPACE in corso...")
file_path = "./space/space_summ.json" # Assicurati che il file sia in questa cartella
space_reviews = []

with open(file_path, 'r', encoding='utf-8') as f:
    # MAGIA QUI: Leggiamo l'intero file in un colpo solo!
    tutti_gli_hotel = json.load(f)
    
    # Ora iteriamo sulla lista di hotel
    for hotel_data in tutti_gli_hotel:
        hotel_id = hotel_data['entity_id']
        
        # Estraiamo SOLO le frasi originali
        for rev in hotel_data['reviews']:
            # Ogni recensione ha una lista di frasi. Le uniamo.
            full_text = " ".join(rev['sentences'])
            space_reviews.append({
                'hotel_id': hotel_id,
                'review_id': rev['review_id'],
                'text': full_text
            })

df_space = pd.DataFrame(space_reviews)
print(f"Dataset SPACE caricato con successo!")
print(f"Hotel unici trovati: {df_space['hotel_id'].nunique()}")
print(f"Totale recensioni da analizzare: {len(df_space)}")

# 3. CARICAMENTO MODELLI (Assicurati di aver incollato le classi prima!)
print("\nCaricamento dei Modelli ACOS (Step 1 e Step 2)...")
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

percorso_pesi_step1 = "./best_multitask_extractor_restaurant/pytorch_model.bin"
percorso_pesi_step2 = "./best_classifier_restaurant/pytorch_model.bin"
numero_categorie_step2 = 13 # 13 per i ristoranti, o 121 se usi i laptop

# (Decommenta e adatta quando hai incollato le classi)
model_step1 = ModernBertACOS_Extractor(num_labels=5).to(device)
model_step1.load_state_dict(torch.load(percorso_pesi_step1, map_location=device))
model_step1.eval()

model_step2 = ModernBertACOSClassifier("./best_model_restaurant", numero_categorie_step2).to(device)
model_step2.load_state_dict(torch.load(percorso_pesi_step2, map_location=device))
model_step2.eval()

print("Pronti per l'estrazione Zero-Shot!")


Caricamento dataset SPACE in corso...
Dataset SPACE caricato con successo!
Hotel unici trovati: 50
Totale recensioni da analizzare: 5000

Caricamento dei Modelli ACOS (Step 1 e Step 2)...


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: ./best_model_restaurant
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pronti per l'estrazione Zero-Shot!


### Estrazione Zero-Shot delle Quadruple ACOS

Questo codice implementa il pipeline principale di estrazione dei quadrupli ACOS per ogni recensione dell'hotel:

**Procedura:**

1. **Caricamento delle categorie**: Legge da file pickle la lista di tutte le categorie disponibili (es. "pulizia", "servizio", "posizione", etc.)

2. **Mapping dei tag BIO**: Definisce il dizionario che traduce gli output del modello Step 1 in label leggibili:
   - `O`: Nessuna entità (Outside)
   - `B-ASP` / `I-ASP`: Inizio/Interno di un aspetto
   - `B-OPI` / `I-OPI`: Inizio/Interno di un'opinione

3. **Loop di estrazione**:
   - Per ogni recensione, chiama `predict_quadruples_space()` che:
     - **Step 1**: Estrae gli aspetti e le opinioni (token classification)
     - **Step 2**: Per ogni coppia (aspetto, opinione), classifica la categoria e il sentiment
   - Per ogni quadrupla estratta, aggiunge i metadati (hotel_id, review_id, testo) e la salva

4. **Gestione errori**: Se una recensione causa crash (testo malformato, encoding issues), la salta silenziosamente e continua

5. **Salvataggio CSV**: Converte tutte le quadruple in un DataFrame e salva il file `quadruple_space_estratte.csv`

**Output**: File CSV con colonne: `hotel_id`, `review_id`, `testo_recensione`, `aspect`, `opinion`, `category`, `sentiment`

In [ ]:
# Lista vuota per salvare i risultati
tutte_le_quadruple = []

df_da_analizzare = df_space 

print(f"Inizio estrazione Zero-Shot su {len(df_da_analizzare)} recensioni (potrebbe volerci un po')...")

with open("data_coppie/restaurant_categories.pkl", "rb") as f:
    category_list = pickle.load(f)
print(f"Categorie caricate: {len(category_list)}")

id2label_dict = {0: 'O', 1: 'B-ASP', 2: 'I-ASP', 3: 'B-OPI', 4: 'I-OPI'}

# Iteriamo su tutte le recensioni con una bella barra di progresso
for index, row in tqdm(df_da_analizzare.iterrows(), total=len(df_da_analizzare), desc="Analisi Recensioni"):
    testo_recensione = row['text']
    hotel_id = row['hotel_id']
    review_id = row['review_id']
    
    try:
        quadruple_estratte = predict_quadruples_space(
            text=testo_recensione,
            model_1=model_step1,
            model_2=model_step2,
            tokenizer=tokenizer,
            cat_list=category_list,
            device=device,
            id2label=id2label_dict,
            best_threshold=0.57
        )
        
        # Per ogni quadrupla trovata, aggiungiamo i metadati dell'hotel e la salviamo
        for q in quadruple_estratte:
            tutte_le_quadruple.append({
                'hotel_id': hotel_id,
                'review_id': review_id,
                'testo_recensione': testo_recensione,
                'aspect': q['aspect_testo'],
                'opinion': q['opinion_testo'],
                'category': q['category'],
                'sentiment': q['sentiment']
            })
            
    except Exception as e:
        # Se una recensione è scritta in modo strano e manda in crash il modello, 
        # la ignoriamo e andiamo avanti senza far bloccare tutto il programma!
        print(f"Errore ignorato nella recensione {review_id}: {e}")

# --- SALVATAGGIO DEI RISULTATI ---
df_risultati_space = pd.DataFrame(tutte_le_quadruple)

# Creiamo il file CSV con tutte le quadruple trovate!
nome_file_output = "quadruple_space_estratte.csv"
df_risultati_space.to_csv(nome_file_output, index=False)

print(f"\nEstrazione completata!")
print(f"Sono state trovate in totale {len(df_risultati_space)} quadruple ACOS.")
print(f"I risultati sono stati salvati al sicuro nel file: {nome_file_output}")

display(df_risultati_space.head())

Inizio estrazione Zero-Shot su 5000 recensioni (potrebbe volerci un po')...
Categorie caricate: 13


Analisi Recensioni: 100%|██████████| 5000/5000 [57:20<00:00,  1.45it/s]  


Estrazione completata! 🎉
Sono state trovate in totale 1433 quadruple ACOS.
I risultati sono stati salvati al sicuro nel file: quadruple_space_estratte.csv


,hotel_id,review_id,testo_recensione,aspect,opinion,category,sentiment
0,100597,UR59977476,We stayed here on a lay over home from Cancun....,staff,friendly,SERVICE#GENERAL,2
1,100597,UR79236677,We stayed at the Double Tree for two nights in...,suite,quiet.,AMBIENCE#GENERAL,2
2,100597,UR79236677,We stayed at the Double Tree for two nights in...,suite,"spacious,",AMBIENCE#GENERAL,2
3,100597,UR96726288,"Despite the enormity of this hotel, it very mu...",staff,exceptional,SERVICE#GENERAL,2
4,100597,UR17574959,We stayed at this hotel for one night before s...,restaurant,spacious,AMBIENCE#GENERAL,2


### Clustering Intelligente di Termini Simili

Questo codice implementa un sistema **multi-strategia** per raggruppare gli aspetti estratti nonostante le variazioni linguistiche.

**Componenti:**

1. **Download NLTK**: Scarica i componenti necessari per la tokenizzazione e lemmatizzazione in inglese

2. **Encoder Sentence Transformers**: Carica il modello `all-MiniLM-L6-v2` per convertire frasi in embedding densi (384-dimensionali)

3. **Funzione `are_similar()`** - Confronta due stringhe con 4 strategie diverse:
   - **Exact**: Confronto case-insensitive diretto (baseline)
   - **Lemma**: Riduce ogni parola alla forma di base (es. "servizi" → "servizio") e confronta
   - **Fuzzy**: Calcola la distanza di Levenshtein tra i token ordinati (tollerante agli errori di battitura)
   - **Embedding**: Codifica i testi in vettori e misura la similarità coseno (coglie il significato semantico)

**Logica:**
- Se uno dei valori è `null` o NaN, ritorna un semplice confronto di uguaglianza
- Normalizza il testo (minuscole, spazi rimossi)
- Ritorna `True` se i testi sono considerati simili secondo la strategia scelta

Questa funzione sarà usata dalla `cluster_terms()` per raggruppare gli aspetti estratti a livello di hotel.

In [ ]:
# Scarichiamo i dizionari di NLTK (lo fa in 2 secondi)
nltk.download('punkt')
nltk.download('punkt_tab') 
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

# Modello encoder raccomandato (piccolo e forte in STS)
encoder = SentenceTransformer('all-MiniLM-L6-v2') 

def are_similar(text1, text2, strategy="exact", threshold=0.75):
    """
    Confronta due stringhe usando la strategia scelta.
    Strategie supportate: 'exact', 'lemma', 'fuzzy', 'embedding'
    """
    if pd.isna(text1) or pd.isna(text2) or text1 == "null" or text2 == "null":
        return text1 == text2

    t1, t2 = str(text1).lower().strip(), str(text2).lower().strip()

    # 1. EXACT MATCH (Baseline)
    if strategy == "exact":
        return t1 == t2

    # 2. LEMMATIZZAZIONE (Ora con NLTK!)
    elif strategy == "lemma":
        lemmas1 = " ".join([lemmatizer.lemmatize(w) for w in word_tokenize(t1)])
        lemmas2 = " ".join([lemmatizer.lemmatize(w) for w in word_tokenize(t2)])
        return lemmas1 == lemmas2

    # 3. FUZZY MATCH (TheFuzz - Distanza di Levenshtein)
    elif strategy == "fuzzy":
        score = fuzz.token_sort_ratio(t1, t2)
        return score >= (threshold * 100)

    # 4. EMBEDDING (Sentence Transformers - Similarità Coseno)
    elif strategy == "embedding":
        emb1 = encoder.encode(t1, convert_to_tensor=True)
        emb2 = encoder.encode(t2, convert_to_tensor=True)
        cosine_score = util.cos_sim(emb1, emb2).item()
        return cosine_score >= threshold

    return False

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/cristinatomaciello/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/cristinatomaciello/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/cristinatomaciello/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Clustering dei Termini Estratti

La funzione `cluster_terms()` raggruppa gli aspetti simili per ridurre ridondanza e rumore nei dati.

**Come funziona:**

1. **Input**: Una lista di termini (es. tutti gli aspetti positivi di un hotel) e una strategia di confronto

2. **Logica di clustering**:
   - Inizializza un dizionario vuoto `clusters` che mappa ogni "termine rappresentante" al suo conteggio
   - Per ogni termine nella lista:
     - Se è `null` o NaN, lo ignora
     - Confronta il termine con i rappresentanti dei cluster già creati usando la strategia scelta
     - Se trova una corrispondenza, incrementa il conteggio del cluster e passa al termine successivo
     - Se non trova match con nessun cluster esistente, crea un nuovo cluster con questo termine come rappresentante

3. **Output**: Un dizionario dove le chiavi sono i termini rappresentanti e i valori sono i conteggi
   - Esempio: `{"pulizia": 45, "servizio": 38, "colazione": 22}`

**Vantaggio**: I termini come "pulito", "pulizia", "limpieza" verranno raggruppati nello stesso cluster a seconda della strategia usata, evitando frammentazione dei dati.

In [ ]:
def cluster_terms(terms_list, strategy="exact", threshold=0.75):
    """
    Prende una lista di termini estratti e li raggruppa in base alla similarità.
    Ritorna un dizionario: { "termine_rappresentante": conteggio }
    """
    clusters = {} # Mappa il "termine_rappresentante" al suo conteggio totale
    
    for term in terms_list:
        if term == "null" or pd.isna(term): 
            continue
            
        found_cluster = False
        
        # Confrontiamo il termine con i rappresentanti dei cluster già creati
        for rep in clusters.keys():
            if are_similar(term, rep, strategy, threshold):
                clusters[rep] += 1
                found_cluster = True
                break
                
        # Se non assomiglia a nessun cluster esistente, ne crea uno nuovo
        if not found_cluster:
            clusters[term] = 1
            
    return clusters

### Aggregazione Multi-Strategia e Generazione dei Prompt

Questo codice esegue l'aggregazione delle quadruple estratte per tutti gli hotel usando 4 strategie diverse di clustering.

**Procedura:**

1. **Caricamento dei dati**: Legge il CSV `quadruple_space_estratte.csv` con tutte le quadruple estratte

2. **Preparazione**: Crea la cartella `prompt_aggregati` dove salvare i risultati

3. **Definizione strategie**: Prepara l'elenco delle 4 strategie di clustering: `exact`, `lemma`, `fuzzy`, `embedding`

4. **Loop su ogni strategia**:
   - Chiama `aggregate_hotel_reviews()` che per ogni hotel:
     - Filtra le quadruple di quel hotel
     - Raggruppa gli aspetti positivi (sentiment=2) usando `cluster_terms()` con la strategia scelta
     - Raggruppa gli aspetti negativi (sentiment=0) nello stesso modo
     - Ordina i cluster dal più al meno menzionato
     - Chiama `verbalizza_statistiche()` per convertire i conteggi in testo narrativo con percentuali
   - Salva il risultato in CSV (`prompt_hotel_{strategia}.csv`)

5. **Output**: 4 file CSV (uno per strategia) contenenti per ogni hotel:
   - `hotel_id`: ID dell'hotel
   - `totale_quadruple_estratte`: numero di quadruple ACOS trovate
   - `prompt_per_llm`: testo narrativo con le statistiche (pronto per l'LLM)

In [ ]:
def verbalizza_statistiche(hotel_id, top_pos, top_neg, totale_utenti):
    """
    Trasforma i conteggi in frasi discorsive con percentuali.
    """
    testo_prompt = f"Statistiche di recensione per l'Hotel {hotel_id}:\n"
    testo_prompt += "="*50 + "\n"
    
    testo_prompt += "ASPETTI POSITIVI:\n"
    for aspect, count in top_pos.items():
        # Calcoliamo la percentuale (Arrotondata)
        percentuale = (count / totale_utenti) * 100
        # Filtriamo il rumore: mostriamo solo gli aspetti menzionati da almeno il 3% degli utenti
        if percentuale >= 3.0: 
            testo_prompt += f"- Il {percentuale:.1f}% degli utenti apprezza: '{aspect}'.\n"
            
    testo_prompt += "\nASPETTI NEGATIVI:\n"
    for aspect, count in top_neg.items():
        percentuale = (count / totale_utenti) * 100
        if percentuale >= 3.0:
            testo_prompt += f"- Il {percentuale:.1f}% degli utenti si lamenta di: '{aspect}'.\n"
            
    return testo_prompt

def aggregate_hotel_reviews(df_estratti, strategy="exact", threshold=0.75, reviews_per_hotel=100):
    """
    Aggrega le quadruple e genera la verbalizzazione statistica per l'LLM.
    """
    print(f"Avvio aggregazione con strategia: '{strategy.upper()}' (Soglia: {threshold})")
    
    hotel_profiles = []
    hotel_ids = df_estratti['hotel_id'].unique()
    
    for h_id in hotel_ids:
        # 1. Filtriamo le quadruple per questo hotel
        df_hotel = df_estratti[df_estratti['hotel_id'] == h_id]
        
        # 2. Estraiamo gli aspetti positivi e negativi
        aspetti_positivi = df_hotel[df_hotel['sentiment'] == 2]['aspect'].tolist()
        aspetti_negativi = df_hotel[df_hotel['sentiment'] == 0]['aspect'].tolist()
        
        # 3. Applichiamo la Fabbrica della Similarità (es. Embedding, Exact, Fuzzy)
        cluster_pos = cluster_terms(aspetti_positivi, strategy, threshold)
        cluster_neg = cluster_terms(aspetti_negativi, strategy, threshold)
        
        # 4. Ordiniamo i cluster dal più menzionato al meno menzionato
        top_pos = dict(sorted(cluster_pos.items(), key=lambda item: item[1], reverse=True))
        top_neg = dict(sorted(cluster_neg.items(), key=lambda item: item[1], reverse=True))
        
        # 5. CREIAMO LA VERBALIZZAZIONE PER L'LLM
        prompt_llm = verbalizza_statistiche(h_id, top_pos, top_neg, totale_utenti=reviews_per_hotel)
        
        hotel_profiles.append({
            'hotel_id': h_id,
            'totale_quadruple_estratte': len(df_hotel),
            'prompt_per_llm': prompt_llm  
        })
        
    return pd.DataFrame(hotel_profiles)

# 1. Carica le quadruple estratte
df_estratti = pd.read_csv("quadruple_space_estratte.csv")

# 2. Creiamo la cartella dove salvare tutto (se non esiste già)
cartella_output = "prompt_aggregati"
os.makedirs(cartella_output, exist_ok=True)
print(f"Cartella di destinazione impostata: '{cartella_output}/'\n")

# 3. Definiamo tutte le strategie che vogliamo testare
strategie_da_provare = ["exact", "lemma", "fuzzy", "embedding"]

# 4. Facciamo girare l'aggregazione per ciascuna strategia e la salviamo!
for strat in strategie_da_provare:
    print(f"Avvio elaborazione per la strategia: {strat.upper()}...")
    
    # Eseguiamo l'aggregazione
    df_aggregato = aggregate_hotel_reviews(df_estratti, strategy=strat, threshold=0.65)
    
    # Creiamo il nome del file e uniamo il percorso della cartella
    nome_file_prompt = f"prompt_hotel_{strat}.csv"
    percorso_completo = os.path.join(cartella_output, nome_file_prompt)
    
    # Salviamo il file DENTRO la cartella
    df_aggregato.to_csv(percorso_completo, index=False)
    print(f"File salvato con successo in: {percorso_completo}\n")

print("TUTTE LE STRATEGIE SONO STATE ELABORATE E SALVATE NELLA CARTELLA!")

Cartella di destinazione impostata: 'prompt_aggregati/'

Avvio elaborazione per la strategia: EXACT...
Avvio aggregazione con strategia: 'EXACT' (Soglia: 0.65)
File salvato con successo in: prompt_aggregati/prompt_hotel_exact.csv

Avvio elaborazione per la strategia: LEMMA...
Avvio aggregazione con strategia: 'LEMMA' (Soglia: 0.65)
File salvato con successo in: prompt_aggregati/prompt_hotel_lemma.csv

Avvio elaborazione per la strategia: FUZZY...
Avvio aggregazione con strategia: 'FUZZY' (Soglia: 0.65)
File salvato con successo in: prompt_aggregati/prompt_hotel_fuzzy.csv

Avvio elaborazione per la strategia: EMBEDDING...
Avvio aggregazione con strategia: 'EMBEDDING' (Soglia: 0.65)
File salvato con successo in: prompt_aggregati/prompt_hotel_embedding.csv

TUTTE LE STRATEGIE SONO STATE ELABORATE E SALVATE NELLA CARTELLA!


In [ ]:
# 1. Carica il CSV con i prompt 
percorso_file = "prompt_aggregati/prompt_hotel_embedding.csv" 
df_prompt = pd.read_csv(percorso_file)

riassunti_finali = []

# Scegli il nome esatto del modello che hai scaricato con Ollama
NOME_MODELLO = 'qwen2.5:7b' 

print(f"Inizio generazione riassunti con {NOME_MODELLO} in locale per {len(df_prompt)} hotel...")

# 2. Ciclo su ogni hotel
for index, row in tqdm(df_prompt.iterrows(), total=len(df_prompt)):
    hotel_id = row['hotel_id']
    statistiche_testo = row['prompt_per_llm'] 
    
    if pd.isna(statistiche_testo) or len(str(statistiche_testo).strip()) < 50:
         riassunti_finali.append({
            'hotel_id': hotel_id,
            'riassunto': "Dati insufficienti per generare un riassunto affidabile."
        })
         continue
    
    prompt_completo = f"""
    Sei un esperto agente di viaggio e copywriter. 
    Leggi le seguenti statistiche aggregate estratte dalle recensioni degli utenti per questo hotel.
    Scrivi un breve riassunto (massimo 3 o 4 frasi) rivolto ai futuri clienti.
    Evidenzia i punti di forza principali e menziona gli eventuali difetti in modo professionale e oggettivo.
    Usa un paragrafo fluido, non usare elenchi puntati.
    REGOLA FONDAMENTALE: DEVI SCRIVERE IL RIASSUNTO ESCLUSIVAMENTE IN LINGUA ITALIANA. NON USARE IL CINESE O L'INGLESE.
    
    {statistiche_testo}
    """
    
    try:
        # CHIAMATA A OLLAMA IN LOCALE!
        response = ollama.chat(model=NOME_MODELLO, messages=[
            {
                'role': 'user',
                'content': prompt_completo,
            },
        ])
        
        testo_generato = response['message']['content'].strip()
        
        riassunti_finali.append({
            'hotel_id': hotel_id,
            'riassunto': testo_generato
        })
        
    except Exception as e:
        print(f"\nErrore con l'hotel {hotel_id}: {e}")
        riassunti_finali.append({
            'hotel_id': hotel_id,
            'riassunto': "Errore durante la generazione."
        })

# 3. Salvataggio finale
df_riassunti = pd.DataFrame(riassunti_finali)
df_riassunti.to_csv("riassunti_finali_qwen.csv", index=False)

print("\nFinito! I riassunti sono stati creati e salvati nel file 'riassunti_finali_qwen.csv'")

print("\n--- ESEMPIO DI RIASSUNTO GENERATO ---")
print(f"Hotel {df_riassunti.iloc[0]['hotel_id']}:")
print(df_riassunti.iloc[0]['riassunto'])

Inizio generazione riassunti con qwen2.5:7b in locale per 50 hotel...


100%|██████████| 50/50 [10:22<00:00, 12.46s/it]


Finito! I riassunti sono stati creati e salvati nel file 'riassunti_finali_qwen.csv'

--- ESEMPIO DI RIASSUNTO GENERATO ---
Hotel 100597:
Gli ospiti dell'Hotel 100597 apprezzano particolarmente il servizio offerto dal personale, secondo il 8,0% delle recensioni. Tuttavia, è importante notare che non sono state registrate critiche specifiche sui servizi offerti, suggerendo un'esperienza generalmente positiva.


In [3]:
NOME_MODELLO = 'qwen2.5:7b' 
strategie = ["exact", "lemma", "fuzzy", "embedding"]

print(f"Avvio della generazione massiva per tutte le strategie con {NOME_MODELLO}...\n")

for strat in strategie:
    percorso_file = f"prompt_aggregati/prompt_hotel_{strat}.csv" 
    
    # Controlliamo se il file esiste davvero per sicurezza
    if not os.path.exists(percorso_file):
        print(f"File non trovato: {percorso_file}. Salto questa strategia.")
        continue
        
    print(f"--- Elaborazione strategia: {strat.upper()} ---")
    df_prompt = pd.read_csv(percorso_file)
    riassunti_finali = []
    
    for index, row in tqdm(df_prompt.iterrows(), total=len(df_prompt)):
        hotel_id = row['hotel_id']
        statistiche_testo = row['prompt_per_llm'] 
        
        if pd.isna(statistiche_testo) or len(str(statistiche_testo).strip()) < 50:
             riassunti_finali.append({
                'hotel_id': hotel_id,
                'riassunto': "Dati insufficienti per generare un riassunto affidabile."
            })
             continue
         
        prompt_completo = f"""
        Sei un esperto agente di viaggio e copywriter. 
        Leggi le seguenti statistiche aggregate estratte dalle recensioni degli utenti per questo hotel.
        Scrivi un breve riassunto (massimo 3 o 4 frasi) rivolto ai futuri clienti.
        Evidenzia i punti di forza principali e menziona gli eventuali difetti in modo professionale e oggettivo.
        Usa un paragrafo fluido, non usare elenchi puntati.
        
        REGOLA FONDAMENTALE: DEVI SCRIVERE IL RIASSUNTO ESCLUSIVAMENTE IN LINGUA ITALIANA. NON USARE IL CINESE O L'INGLESE.
        
        {statistiche_testo}
        """
        
        try:
            response = ollama.chat(model=NOME_MODELLO, messages=[
                {'role': 'user', 'content': prompt_completo}
            ])
            testo_generato = response['message']['content'].strip()
            
            riassunti_finali.append({
                'hotel_id': hotel_id,
                'riassunto': testo_generato
            })
            
        except Exception as e:
            print(f"Errore con l'hotel {hotel_id}: {e}")
            riassunti_finali.append({
                'hotel_id': hotel_id,
                'riassunto': "Errore durante la generazione."
            })

    # Creiamo la cartella di destinazione (se non esiste già)
    cartella_output_riassunti = "riassunti_finali"
    os.makedirs(cartella_output_riassunti, exist_ok=True)

    # Creiamo il nome del file e lo uniamo al percorso della cartella
    nome_file = f"riassunti_{strat}_{NOME_MODELLO.replace(':', '_')}.csv"
    percorso_completo = os.path.join(cartella_output_riassunti, nome_file)

    df_riassunti = pd.DataFrame(riassunti_finali)
    df_riassunti.to_csv(percorso_completo, index=False)
    
    print(f"Salvato: {percorso_completo}\n")

print("Calcolo completato per tutte le strategie! I riassunti sono stati salvati nella cartella 'riassunti_finali/'.")

Avvio della generazione massiva per tutte le strategie con qwen2.5:7b...

--- Elaborazione strategia: EXACT ---


100%|██████████| 50/50 [10:01<00:00, 12.02s/it]


Salvato: riassunti_finali/riassunti_exact_qwen2.5_7b.csv

--- Elaborazione strategia: LEMMA ---


100%|██████████| 50/50 [08:58<00:00, 10.77s/it]


Salvato: riassunti_finali/riassunti_lemma_qwen2.5_7b.csv

--- Elaborazione strategia: FUZZY ---


100%|██████████| 50/50 [09:01<00:00, 10.84s/it]


Salvato: riassunti_finali/riassunti_fuzzy_qwen2.5_7b.csv

--- Elaborazione strategia: EMBEDDING ---


100%|██████████| 50/50 [09:01<00:00, 10.83s/it]

Salvato: riassunti_finali/riassunti_embedding_qwen2.5_7b.csv

Calcolo completato per tutte le strategie! I riassunti sono stati salvati nella cartella 'riassunti_finali/'.
